In [1]:
# ============================================================
# FINAL GIF CAPTION SYSTEM - COMPLETE INTEGRATION
# ============================================================

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import pandas as pd
import pickle
from nltk.tokenize import word_tokenize

print("=" * 60)
print("🎬 COMPLETE GIF CAPTION SYSTEM")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

# ============================================================
# 1. LOAD VOCABULARY
# ============================================================

class Vocabulary:
    def __init__(self, freq_threshold=5):
        self.itos = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.stoi = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.freq_threshold = freq_threshold
    
    def __len__(self):
        return len(self.itos)
    
    def numericalize(self, caption):
        tokens = word_tokenize(caption.lower())
        return [self.stoi.get(token, self.stoi["<UNK>"]) for token in tokens]

with open('vocab.pkl', 'rb') as f:
    vocab = pickle.load(f)

print(f"✅ Vocabulary loaded: {len(vocab)} words")

# ============================================================
# 2. LOAD EMOTION MODEL
# ============================================================

emotion_groups = ['contempt', 'negative_intense', 'negative_subdued',
                 'positive_calm', 'positive_energetic', 'surprise']

class GroupedEmotionClassifier(nn.Module):
    def __init__(self, num_classes=6, pretrained=False):
        super(GroupedEmotionClassifier, self).__init__()
        
        resnet = models.resnet50(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

emotion_model = GroupedEmotionClassifier(num_classes=6)
emotion_model.load_state_dict(torch.load('best_model_grouped.pth', map_location=device))
emotion_model = emotion_model.to(device)
emotion_model.eval()

print(f"✅ Emotion model loaded")

# ============================================================
# 3. LOAD LSTM CAPTION MODEL
# ============================================================

class CaptionLSTM(nn.Module):
    def __init__(self, feature_dim, embed_dim, hidden_dim, vocab_size, num_layers=2):
        super(CaptionLSTM, self).__init__()
        
        self.embed_dim = embed_dim
        self.hidden_dim = hidden_dim
        
        self.feature_fc = nn.Linear(feature_dim, embed_dim)
        self.word_embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(0.5)
    
    def forward(self, features, captions):
        caption_embedded = self.word_embed(captions[:, :-1])
        features_embedded = self.feature_fc(features).unsqueeze(1)
        embeddings = torch.cat([features_embedded, caption_embedded], dim=1)
        lstm_out, _ = self.lstm(embeddings)
        lstm_out = lstm_out[:, 1:, :]
        lstm_out = self.dropout(lstm_out)
        outputs = self.fc_out(lstm_out)
        return outputs
    
    def generate_caption(self, features, max_len=20):
        self.eval()
        with torch.no_grad():
            caption = [vocab.stoi["<SOS>"]]
            features_embedded = self.feature_fc(features).unsqueeze(1)
            hidden = None
            
            for _ in range(max_len):
                word_tensor = torch.LongTensor([caption[-1]]).to(features.device)
                word_embedded = self.word_embed(word_tensor).unsqueeze(1)
                
                if len(caption) == 1:
                    lstm_input = features_embedded
                else:
                    lstm_input = word_embedded
                
                lstm_out, hidden = self.lstm(lstm_input, hidden)
                output = self.fc_out(lstm_out.squeeze(1))
                predicted = output.argmax(1).item()
                
                caption.append(predicted)
                
                if predicted == vocab.stoi["<EOS>"]:
                    break
            
            caption_words = [vocab.itos[idx] for idx in caption[1:-1]]
            return ' '.join(caption_words)

caption_model = CaptionLSTM(
    feature_dim=2048,
    embed_dim=256,
    hidden_dim=512,
    vocab_size=len(vocab),
    num_layers=2
).to(device)

caption_model.load_state_dict(torch.load('best_caption_model.pth', map_location=device))
caption_model.eval()

print(f"✅ Caption LSTM loaded\n")

print("=" * 60)
print("✅ ALL MODELS LOADED!")
print("=" * 60)

🎬 COMPLETE GIF CAPTION SYSTEM
Device: cpu

✅ Vocabulary loaded: 10322 words
✅ Emotion model loaded
✅ Caption LSTM loaded

✅ ALL MODELS LOADED!


In [2]:
# ============================================================
# GIF FEATURE EXTRACTION WITH EMOTION CONDITIONING
# ============================================================

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                       std=[0.229, 0.224, 0.225])
])

def extract_middle_frame(gif_path):
    """Extract middle frame from GIF"""
    try:
        with Image.open(gif_path) as gif:
            n_frames = 0
            try:
                while True:
                    gif.seek(n_frames)
                    n_frames += 1
            except EOFError:
                pass
            gif.seek(n_frames // 2)
            return gif.convert('RGB')
    except:
        return None

def get_emotion_features(gif_path):
    """
    Extract emotion features from GIF
    Returns: emotion label, confidence, feature vector (512-dim)
    """
    frame = extract_middle_frame(gif_path)
    if frame is None:
        return None, None, None
    
    frame_tensor = transform(frame).unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Get intermediate features (before final classification)
        features = emotion_model.features(frame_tensor)
        features_flat = features.view(features.size(0), -1)  # 2048-dim
        
        # Get emotion prediction
        output = emotion_model.classifier(features_flat)
        probs = torch.softmax(output, dim=1)
        emotion_idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0, emotion_idx].item()
        
        # Get 512-dim emotion-specific features (after first FC layer)
        emotion_features = emotion_model.classifier[:4](features_flat)  # Up to ReLU
    
    emotion = emotion_groups[emotion_idx]
    
    return emotion, confidence, emotion_features

print("✅ Emotion feature extractor ready")

✅ Emotion feature extractor ready


In [3]:
# ============================================================
# FEATURE FUSION: Emotion → Caption Features
# ============================================================

class EmotionFeatureFusion(nn.Module):
    """
    Projects 512-dim emotion features to 2048-dim for LSTM input
    This allows emotion information to influence caption generation
    """
    def __init__(self):
        super(EmotionFeatureFusion, self).__init__()
        self.projection = nn.Linear(512, 2048)
        self.relu = nn.ReLU()
    
    def forward(self, emotion_features):
        return self.relu(self.projection(emotion_features))

fusion_model = EmotionFeatureFusion().to(device)
fusion_model.eval()

print("✅ Feature fusion module ready")

✅ Feature fusion module ready


In [4]:
# ============================================================
# COMPLETE GIF → CAPTION PIPELINE
# ============================================================

def generate_caption_for_gif(gif_path):
    """
    Complete pipeline: GIF → Emotion-conditioned Caption
    
    Returns:
        caption: Generated caption string
        emotion: Detected emotion
        confidence: Emotion detection confidence
    """
    # Step 1: Get emotion features
    emotion, confidence, emotion_features = get_emotion_features(gif_path)
    
    if emotion is None:
        return "Unable to process GIF", None, None
    
    # Step 2: Project emotion features to caption feature space
    with torch.no_grad():
        caption_features = fusion_model(emotion_features)  # 512 → 2048
    
    # Step 3: Generate caption using LSTM
    caption = caption_model.generate_caption(caption_features)
    
    return caption, emotion, confidence

print("=" * 60)
print("✅ COMPLETE PIPELINE READY!")
print("=" * 60)

✅ COMPLETE PIPELINE READY!


In [5]:
# ============================================================
# TEST ON GIFGIF DATASET
# ============================================================

# Load test data
test_df = pd.read_csv('test_grouped.csv')

print("\n🧪 Testing on GIFGIF dataset...")
print("=" * 60)

# Test on 10 random samples
samples = test_df.sample(10, random_state=42)

results = []

for idx, (_, row) in enumerate(samples.iterrows()):
    gif_path = row['gif_path']
    true_emotion = row['emotion_group']
    gif_id = row['gif_id']
    
    # Generate caption
    caption, pred_emotion, confidence = generate_caption_for_gif(gif_path)
    
    match = "✅" if pred_emotion == true_emotion else "❌"
    
    print(f"\n{'─' * 60}")
    print(f"  GIF #{idx+1}: {gif_id}")
    print(f"  {match} True Emotion:      {true_emotion}")
    print(f"     Predicted Emotion:  {pred_emotion} ({confidence:.1%})")
    print(f"  💬 Generated Caption: {caption}")
    print(f"{'─' * 60}")
    
    results.append({
        'gif_id': gif_id,
        'true_emotion': true_emotion,
        'pred_emotion': pred_emotion,
        'confidence': confidence,
        'caption': caption,
        'emotion_correct': pred_emotion == true_emotion
    })

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('caption_results_sample.csv', index=False)

print("\n" + "=" * 60)
print("✅ TESTING COMPLETE!")
print("=" * 60)
print(f"Emotion accuracy: {results_df['emotion_correct'].mean():.1%}")
print(f"Results saved to: caption_results_sample.csv")


🧪 Testing on GIFGIF dataset...

────────────────────────────────────────────────────────────
  GIF #1: HSNAwKc63r9lK
  ✅ True Emotion:      negative_intense
     Predicted Emotion:  negative_intense (56.1%)
  💬 Generated Caption: stocking of a cat sitting on a chair .
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #2: 9YdllfJmcs5RS
  ✅ True Emotion:      negative_intense
     Predicted Emotion:  negative_intense (96.6%)
  💬 Generated Caption: paperback a large clock .
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #3: aF8nZCmUEgmBO
  ❌ True Emotion:      surprise
     Predicted Emotion:  negative_intense (45.4%)
  💬 Generated Caption: mustache a cat sitting on a chair .
────────────────────────────────────────────────────────────

────────────────────────────────────────────────────────────
  GIF #4: HmsAidDdKMTle
  ❌ True 